<a href="https://colab.research.google.com/github/xiyuan1avery/ma2288/blob/research-v2/notebooks/14_v2_collect_onpolicy_rollouts.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q transformers

In [2]:
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

SEED = 42

torch.manual_seed(SEED)
np.random.seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "cpu"
)

print("Device:", device)

Device: cuda


In [3]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [4]:
DRIVE_DIRECTORY = Path(
    "/content/drive/MyDrive/ma2288_nextlat"
)

TRAIN_DATA_PATH = (
    DRIVE_DIRECTORY
    / "train_teacher_states.pt"
)

MULTISTEP_CHECKPOINT_PATH = (
    DRIVE_DIRECTORY
    / "checkpoints"
    / "multistep_transition_seed42.pt"
)

print("Train data:", TRAIN_DATA_PATH.exists())
print(
    "Checkpoint:",
    MULTISTEP_CHECKPOINT_PATH.exists(),
)

Train data: True
Checkpoint: True


In [5]:
train_artifact = torch.load(
    TRAIN_DATA_PATH,
    map_location="cpu",
    weights_only=False,
)

train_token_ids = train_artifact[
    "token_ids"
]

train_teacher_hidden = train_artifact[
    "hidden_states"
]

MODEL_NAME = train_artifact[
    "model_name"
]

print("Tokens:", train_token_ids.shape)
print(
    "Teacher hidden:",
    train_teacher_hidden.shape,
)

Tokens: torch.Size([400, 64])
Teacher hidden: torch.Size([400, 64, 768])


In [6]:
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
)

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)

target_model = (
    AutoModelForCausalLM
    .from_pretrained(MODEL_NAME)
    .to(device)
)

target_model.eval()

for parameter in target_model.parameters():
    parameter.requires_grad = False

token_embedding = (
    target_model.get_input_embeddings()
)

print("Target model loaded.")

config.json:   0%|          | 0.00/762 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  353MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Target model loaded.


In [7]:
class ResidualTransitionMLP(nn.Module):
    def __init__(
        self,
        hidden_dimension=768,
        bottleneck_dimension=512,
    ):
        super().__init__()

        combined_dimension = (
            hidden_dimension * 2
        )

        self.input_normalization = nn.LayerNorm(
            combined_dimension
        )

        self.network = nn.Sequential(
            nn.Linear(
                combined_dimension,
                bottleneck_dimension,
            ),
            nn.GELU(),
            nn.Linear(
                bottleneck_dimension,
                hidden_dimension,
            ),
        )

    def forward(
        self,
        current_hidden,
        next_token_embedding,
    ):
        combined = torch.cat(
            [
                current_hidden,
                next_token_embedding,
            ],
            dim=-1,
        )

        delta = self.network(
            self.input_normalization(
                combined
            )
        )

        return current_hidden + delta

In [8]:
checkpoint = torch.load(
    MULTISTEP_CHECKPOINT_PATH,
    map_location="cpu",
    weights_only=False,
)

behavior_model = ResidualTransitionMLP(
    hidden_dimension=checkpoint[
        "hidden_dimension"
    ],
    bottleneck_dimension=checkpoint[
        "bottleneck_dimension"
    ],
)

behavior_model.load_state_dict(
    checkpoint["model_state_dict"]
)

behavior_model = behavior_model.to(device)
behavior_model.eval()

print("Behavior model loaded.")

Behavior model loaded.


In [9]:
PREFIX_LENGTH = 32
START_POSITION = PREFIX_LENGTH - 1

ROLLOUT_LENGTH = 8
TRIALS_PER_PREFIX = 2
NUMBER_OF_PREFIXES = 400

print(
    "Expected chains:",
    NUMBER_OF_PREFIXES
    * TRIALS_PER_PREFIX,
)

Expected chains: 800


In [10]:
def generate_onpolicy_draft(
    initial_hidden,
    rollout_length,
    sampling_seed,
):
    torch.manual_seed(
        sampling_seed
    )

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(
            sampling_seed
        )

    current_hidden = initial_hidden

    token_list = []
    q_list = []
    predicted_state_list = []

    with torch.no_grad():
        for step in range(
            rollout_length
        ):
            logits = target_model.lm_head(
                current_hidden
            )

            q = torch.softmax(
                logits,
                dim=-1,
            )

            token = torch.multinomial(
                q,
                num_samples=1,
            )

            token_list.append(token)
            q_list.append(q)

            next_embedding = (
                token_embedding(token)
                .squeeze(1)
            )

            current_hidden = (
                behavior_model(
                    current_hidden,
                    next_embedding,
                )
            )

            predicted_state_list.append(
                current_hidden
            )

    return (
        torch.cat(
            token_list,
            dim=1,
        ),
        torch.stack(
            q_list,
            dim=1,
        ),
        torch.stack(
            predicted_state_list,
            dim=1,
        ),
    )

In [11]:
def get_target_labels(
    prefix_ids,
    draft_tokens,
):
    full_sequence = torch.cat(
        [
            prefix_ids,
            draft_tokens,
        ],
        dim=1,
    )

    prefix_length = (
        prefix_ids.shape[1]
    )

    rollout_length = (
        draft_tokens.shape[1]
    )

    with torch.no_grad():
        outputs = target_model(
            input_ids=full_sequence,
            output_hidden_states=True,
            return_dict=True,
        )

    # p_1 ... p_d：预测每个 draft token 的分布
    target_logits = outputs.logits[
        :,
        prefix_length - 1:
        prefix_length + rollout_length - 1,
        :,
    ]

    target_probabilities = torch.softmax(
        target_logits,
        dim=-1,
    )

    # h_(t+1) ... h_(t+d)
    target_hidden_states = (
        outputs.hidden_states[-1][
            :,
            prefix_length:
            prefix_length + rollout_length,
            :,
        ]
    )

    return (
        target_probabilities,
        target_hidden_states,
    )

In [12]:
initial_hidden_rows = []
draft_token_rows = []
target_hidden_rows = []
behavior_hidden_rows = []
output_kl_rows = []
acceptance_probability_rows = []
prompt_index_rows = []
trial_index_rows = []

In [13]:
import time

collection_start = time.time()

for prompt_index in range(
    NUMBER_OF_PREFIXES
):
    prefix_ids = (
        train_token_ids[
            prompt_index,
            :PREFIX_LENGTH,
        ]
        .unsqueeze(0)
        .to(device)
    )

    initial_hidden = (
        train_teacher_hidden[
            prompt_index,
            START_POSITION,
        ]
        .unsqueeze(0)
        .float()
        .to(device)
    )

    for trial_index in range(
        TRIALS_PER_PREFIX
    ):
        sampling_seed = (
            SEED
            + prompt_index * 100
            + trial_index
        )

        (
            draft_tokens,
            q_probabilities,
            behavior_hidden_states,
        ) = generate_onpolicy_draft(
            initial_hidden=initial_hidden,
            rollout_length=(
                ROLLOUT_LENGTH
            ),
            sampling_seed=(
                sampling_seed
            ),
        )

        (
            target_probabilities,
            target_hidden_states,
        ) = get_target_labels(
            prefix_ids=prefix_ids,
            draft_tokens=draft_tokens,
        )

        p_log = torch.log(
            target_probabilities
            .clamp_min(1e-12)
        )

        q_log = torch.log(
            q_probabilities
            .clamp_min(1e-12)
        )

        output_kl = torch.sum(
            target_probabilities
            * (p_log - q_log),
            dim=-1,
        )

        selected_p = (
            target_probabilities.gather(
                dim=2,
                index=(
                    draft_tokens
                    .unsqueeze(-1)
                ),
            )
            .squeeze(-1)
        )

        selected_q = (
            q_probabilities.gather(
                dim=2,
                index=(
                    draft_tokens
                    .unsqueeze(-1)
                ),
            )
            .squeeze(-1)
        )

        acceptance_probability = (
            torch.minimum(
                torch.ones_like(
                    selected_p
                ),
                selected_p
                / selected_q.clamp_min(
                    1e-12
                ),
            )
        )

        initial_hidden_rows.append(
            initial_hidden
            .detach()
            .cpu()
            .half()
        )

        draft_token_rows.append(
            draft_tokens
            .detach()
            .cpu()
        )

        target_hidden_rows.append(
            target_hidden_states
            .detach()
            .cpu()
            .half()
        )

        behavior_hidden_rows.append(
            behavior_hidden_states
            .detach()
            .cpu()
            .half()
        )

        output_kl_rows.append(
            output_kl
            .detach()
            .cpu()
            .half()
        )

        acceptance_probability_rows.append(
            acceptance_probability
            .detach()
            .cpu()
            .half()
        )

        prompt_index_rows.append(
            prompt_index
        )

        trial_index_rows.append(
            trial_index
        )

    if (
        prompt_index == 0
        or (prompt_index + 1) % 50 == 0
    ):
        elapsed = (
            time.time()
            - collection_start
        )

        print(
            f"Collected "
            f"{prompt_index + 1}/"
            f"{NUMBER_OF_PREFIXES} "
            f"prefixes in "
            f"{elapsed:.1f} seconds"
        )

Collected 1/400 prefixes in 1.7 seconds
Collected 50/400 prefixes in 4.4 seconds
Collected 100/400 prefixes in 8.1 seconds
Collected 150/400 prefixes in 9.9 seconds
Collected 200/400 prefixes in 11.8 seconds
Collected 250/400 prefixes in 13.4 seconds
Collected 300/400 prefixes in 15.1 seconds
Collected 350/400 prefixes in 17.1 seconds
Collected 400/400 prefixes in 21.4 seconds


In [14]:
onpolicy_artifact = {
    "model_name": MODEL_NAME,
    "behavior_checkpoint": (
        "multistep_transition_seed42.pt"
    ),
    "prefix_length": PREFIX_LENGTH,
    "rollout_length": ROLLOUT_LENGTH,
    "trials_per_prefix": (
        TRIALS_PER_PREFIX
    ),
    "initial_hidden": torch.cat(
        initial_hidden_rows,
        dim=0,
    ),
    "draft_token_ids": torch.cat(
        draft_token_rows,
        dim=0,
    ),
    "target_hidden_states": torch.cat(
        target_hidden_rows,
        dim=0,
    ),
    "behavior_hidden_states": torch.cat(
        behavior_hidden_rows,
        dim=0,
    ),
    "behavior_output_kl": torch.cat(
        output_kl_rows,
        dim=0,
    ),
    "behavior_acceptance_probability": (
        torch.cat(
            acceptance_probability_rows,
            dim=0,
        )
    ),
    "prompt_indices": torch.tensor(
        prompt_index_rows
    ),
    "trial_indices": torch.tensor(
        trial_index_rows
    ),
}

In [15]:
for key, value in (
    onpolicy_artifact.items()
):
    if isinstance(value, torch.Tensor):
        print(
            key,
            value.shape,
            value.dtype,
        )

initial_hidden torch.Size([800, 768]) torch.float16
draft_token_ids torch.Size([800, 8]) torch.int64
target_hidden_states torch.Size([800, 8, 768]) torch.float16
behavior_hidden_states torch.Size([800, 8, 768]) torch.float16
behavior_output_kl torch.Size([800, 8]) torch.float16
behavior_acceptance_probability torch.Size([800, 8]) torch.float16
prompt_indices torch.Size([800]) torch.int64
trial_indices torch.Size([800]) torch.int64


In [16]:
print(
    "Contains hidden NaN:",
    torch.isnan(
        onpolicy_artifact[
            "target_hidden_states"
        ]
    ).any().item(),
)

print(
    "Mean behavior KL by step:"
)

print(
    onpolicy_artifact[
        "behavior_output_kl"
    ]
    .float()
    .mean(dim=0)
)

print(
    "Mean acceptance probability by step:"
)

print(
    onpolicy_artifact[
        "behavior_acceptance_probability"
    ]
    .float()
    .mean(dim=0)
)

Contains hidden NaN: False
Mean behavior KL by step:
tensor([2.3991e-08, 1.4224e+00, 1.9619e+00, 2.1278e+00, 2.1871e+00, 2.2618e+00,
        2.2290e+00, 2.1557e+00])
Mean acceptance probability by step:
tensor([0.9999, 0.4650, 0.3919, 0.3719, 0.3728, 0.3834, 0.3859, 0.3683])


In [17]:
V2_DATA_DIRECTORY = (
    DRIVE_DIRECTORY
    / "data_v2"
)

V2_DATA_DIRECTORY.mkdir(
    parents=True,
    exist_ok=True,
)

ONPOLICY_DATA_PATH = (
    V2_DATA_DIRECTORY
    / "onpolicy_rollouts_seed42.pt"
)

torch.save(
    onpolicy_artifact,
    ONPOLICY_DATA_PATH,
)

print("Saved:", ONPOLICY_DATA_PATH)
print(
    "File size MB:",
    ONPOLICY_DATA_PATH.stat().st_size
    / 1024**2,
)

Saved: /content/drive/MyDrive/ma2288_nextlat/data_v2/onpolicy_rollouts_seed42.pt
File size MB: 20.011164665222168


In [18]:
loaded_artifact = torch.load(
    ONPOLICY_DATA_PATH,
    map_location="cpu",
    weights_only=False,
)

print(
    "Loaded draft tokens:",
    loaded_artifact[
        "draft_token_ids"
    ].shape,
)

print(
    "Loaded target hidden:",
    loaded_artifact[
        "target_hidden_states"
    ].shape,
)

Loaded draft tokens: torch.Size([800, 8])
Loaded target hidden: torch.Size([800, 8, 768])
